# 1. Advanced RAG Mastery: Retrieval-Augmented Generation

Welcome to the **RAG Mastery** interactive notebook.
Retrieval-Augmented Generation (RAG) combines the parametric memory of Large Language Models (LLMs) with non-parametric external knowledge stores.

In this notebook, we cover end-to-end RAG concepts:
1. **Document Loading & Chunking Strategies** (Fixed-size vs Overlapping vs Semantic)
2. **Dense & Sparse Embedding Generation**
3. **Vector Similarity Metrics & Retrieval** (Cosine Similarity, Dot Product, Euclidean Distance)
4. **Re-Ranking & Hybrid Search** (Combining Sparse BM25 + Dense Vectors with Reciprocal Rank Fusion)
5. **RAG Evaluation Framework** (Faithfulness, Context Precision, Context Recall metrics)

In [ ]:
import math
import re
from dataclasses import dataclass
from typing import List, Dict, Tuple, Any

print("=== RAG Mastery Setup Complete ===")

## Step 1: Chunking Strategies

Effective chunking breaks large documents into optimal context segments for search indexing.
- **Fixed-Size Chunking**: Chunks text by character/word count. Simple, but can split sentences mid-thought.
- **Overlapping Chunking**: Adds sliding window overlap between chunks to preserve context across boundaries.
- **Semantic Chunking**: Splits text on natural boundaries like paragraphs or structural headings.

In [ ]:
class DocumentChunker:
    @staticmethod
    def fixed_size_chunking(text: str, chunk_size: int = 100, overlap: int = 20) -> List[str]:
        words = text.split()
        chunks = []
        start = 0
        while start < len(words):
            end = min(start + chunk_size, len(words))
            chunk = " ".join(words[start:end])
            chunks.append(chunk)
            if end == len(words):
                break
            start += (chunk_size - overlap)
        return chunks

    @staticmethod
    def paragraph_chunking(text: str) -> List[str]:
        paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
        return paragraphs

sample_doc = """Retrieval-Augmented Generation (RAG) is an AI framework that retrieves relevant documents from an external knowledge base.
This grounds the model responses on verified context, reducing hallucinations and providing updated information.

Document chunking is the process of splitting text into small, meaningful pieces. Choosing the right chunk size balances contextual completeness with retriever accuracy.

Embedding models convert text chunks into high-dimensional numerical vectors. These vectors capture semantic meaning in mathematical space."""

fixed_chunks = DocumentChunker.fixed_size_chunking(sample_doc, chunk_size=25, overlap=5)
paragraph_chunks = DocumentChunker.paragraph_chunking(sample_doc)

print(f"Fixed Chunks count: {len(fixed_chunks)}")
for i, c in enumerate(fixed_chunks[:2]):
    print(f" Chunk {i+1} ({len(c.split())} words): '{c[:60]}...'")

print(f"\nParagraph Chunks count: {len(paragraph_chunks)}")
for i, c in enumerate(paragraph_chunks):
    print(f" Paragraph {i+1}: '{c[:60]}...'")

## Step 2: Dense & Sparse Embeddings

- **Dense Embeddings**: High-dimensional continuous numerical vectors (e.g. 384d or 1536d) representing semantic concepts.
- **Sparse Embeddings (BM25 / TF-IDF)**: High-dimensional sparse keyword vectors based on term frequencies and inverse document frequencies.

In [ ]:
class SparseBM25:
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus: List[List[str]] = []
        self.doc_lens: List[int] = []
        self.avgdl: float = 0.0
        self.idf: Dict[str, float] = {}

    def fit(self, docs: List[str]):
        self.corpus = [re.findall(r'\w+', doc.lower()) for doc in docs]
        self.doc_lens = [len(doc) for doc in self.corpus]
        self.avgdl = sum(self.doc_lens) / len(self.doc_lens) if self.doc_lens else 1.0
        
        doc_count = len(docs)
        freq_map: Dict[str, int] = {}
        for doc in self.corpus:
            unique_terms = set(doc)
            for term in unique_terms:
                freq_map[term] = freq_map.get(term, 0) + 1
                
        for term, n_q in freq_map.items():
            self.idf[term] = math.log((doc_count - n_q + 0.5) / (n_q + 0.5) + 1.0)

    def search(self, query: str, top_k: int = 3) -> List[Tuple[int, float]]:
        q_terms = re.findall(r'\w+', query.lower())
        scores = []
        for idx, doc in enumerate(self.corpus):
            score = 0.0
            doc_len = self.doc_lens[idx]
            for q in q_terms:
                if q in self.idf:
                    tf = doc.count(q)
                    denom = tf + self.k1 * (1 - self.b + self.b * (doc_len / self.avgdl))
                    score += self.idf[q] * ((tf * (self.k1 + 1)) / denom)
            scores.append((idx, score))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

class SyntheticDenseEncoder:
    """Deterministic synthetic dense embedding generator mapping text to 8-dim vector."""
    @staticmethod
    def encode(text: str) -> List[float]:
        words = re.findall(r'\w+', text.lower())
        dims = [0.0] * 8
        for i, word in enumerate(words):
            val = sum(ord(c) for c in word)
            dims[i % 8] += val
        # L2 Normalize
        norm = math.sqrt(sum(v*v for v in dims)) or 1.0
        return [v / norm for v in dims]

documents = [
    "RAG retrieves relevant context from external documents for LLM prompting.",
    "BM25 is a sparse term-matching algorithm for keyword document retrieval.",
    "Dense embeddings represent semantic text similarity in vector space.",
    "Re-ranking improves retrieval precision by re-scoring top candidates with cross-encoders."
]

bm25 = SparseBM25()
bm25.fit(documents)
sparse_results = bm25.search("retrieval precision", top_k=2)

print("BM25 Keyword Search Results:")
for doc_idx, score in sparse_results:
    print(f" Doc {doc_idx} (Score: {score:.4f}): '{documents[doc_idx]}'")

dense_emb = SyntheticDenseEncoder.encode(documents[0])
print(f"\nSample Dense Embedding Vector (8-dim): {[round(x, 3) for x in dense_emb]}")

## Step 3: Vector Similarity Search & Indexing

Vector search measures similarity between query vectors and document vectors using mathematical metrics:
1. **Cosine Similarity**: Measures directional alignment: $\cos(\theta) = \frac{A \cdot B}{\|A\| \|B\|}
2. **Euclidean Distance**: Measures geometric distance: $d(A, B) = \sqrt{\sum (A_i - B_i)^2}$
3. **Dot Product**: Inner product of vectors (equivalent to cosine similarity for normalized vectors).

In [ ]:
class VectorStore:
    def __init__(self):
        self.vectors: List[List[float]] = []
        self.metadata: List[str] = []

    def add_documents(self, docs: List[str]):
        for doc in docs:
            vec = SyntheticDenseEncoder.encode(doc)
            self.vectors.append(vec)
            self.metadata.append(doc)

    def search(self, query: str, metric: str = "cosine", top_k: int = 3) -> List[Tuple[str, float]]:
        q_vec = SyntheticDenseEncoder.encode(query)
        results = []
        
        for idx, doc_vec in enumerate(self.vectors):
            if metric == "cosine":
                dot = sum(a * b for a, b in zip(q_vec, doc_vec))
                mag_a = math.sqrt(sum(a * a for a in q_vec))
                mag_b = math.sqrt(sum(b * b for b in doc_vec))
                score = dot / (mag_a * mag_b) if mag_a and mag_b else 0.0
            elif metric == "euclidean":
                dist = math.sqrt(sum((a - b) ** 2 for a, b in zip(q_vec, doc_vec)))
                score = 1.0 / (1.0 + dist) # inverted so higher is closer
            results.append((self.metadata[idx], score))
            
        results.sort(key=lambda x: x[1], reverse=True)
        return results[:top_k]

vstore = VectorStore()
vstore.add_documents(documents)
dense_results = vstore.search("semantic similarity vector search", metric="cosine", top_k=2)

print("Dense Vector Search Results (Cosine Similarity):")
for doc_text, score in dense_results:
    print(f" Score {score:.4f}: '{doc_text}'")

## Step 4: Hybrid Search & Re-ranking (RRF)

**Hybrid Search** combines the strengths of Keyword Search (BM25 - exact matching) and Vector Search (Dense - semantic matching).
We use **Reciprocal Rank Fusion (RRF)** to combine rank ordering:
$$RRF\_Score(d) = \sum_{m \in M} \frac{1}{k + r_m(d)}$$
where $k=60$ is a standard smoothing constant.

In [ ]:
class HybridSearchEngine:
    def __init__(self, docs: List[str]):
        self.docs = docs
        self.bm25 = SparseBM25()
        self.bm25.fit(docs)
        self.vstore = VectorStore()
        self.vstore.add_documents(docs)

    def hybrid_search(self, query: str, top_k: int = 3, rrf_k: int = 60) -> List[Tuple[str, float]]:
        # 1. Sparse search
        sparse_res = self.bm25.search(query, top_k=len(self.docs))
        sparse_ranks = {self.docs[idx]: rank + 1 for rank, (idx, _) in enumerate(sparse_res)}

        # 2. Dense search
        dense_res = self.vstore.search(query, metric="cosine", top_k=len(self.docs))
        dense_ranks = {doc: rank + 1 for rank, (doc, _) in enumerate(dense_res)}

        # 3. Reciprocal Rank Fusion
        rrf_scores = {}
        for doc in self.docs:
            r_sparse = sparse_ranks.get(doc, len(self.docs) + 1)
            r_dense = dense_ranks.get(doc, len(self.docs) + 1)
            score = (1.0 / (rrf_k + r_sparse)) + (1.0 / (rrf_k + r_dense))
            rrf_scores[doc] = score

        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_docs[:top_k]

hybrid_engine = HybridSearchEngine(documents)
hybrid_results = hybrid_engine.hybrid_search("RAG keyword retrieval and dense embeddings", top_k=3)

print("Hybrid Search RRF Fusion Results:")
for rank, (doc, score) in enumerate(hybrid_results, 1):
    print(f" Rank {rank} (RRF Score: {score:.5f}): '{doc}'")

## Step 5: RAG Evaluation Metrics

To evaluate RAG pipeline quality, we compute three key metrics:
1. **Faithfulness**: Proportion of claims in the generated answer that are directly supported by retrieved context.
2. **Context Precision**: Ratio of relevant retrieved documents to total retrieved documents in top-K.
3. **Context Recall**: Ratio of relevant retrieved ground-truth facts present in retrieved context.

In [ ]:
class RAGEvaluator:
    @staticmethod
    def evaluate(query: str, retrieved_contexts: List[str], generated_answer: str, ground_truth: str) -> Dict[str, float]:
        # 1. Context Precision: How many retrieved contexts mention query keywords?
        q_keywords = set(re.findall(r'\w+', query.lower()))
        relevant_contexts = 0
        for ctx in retrieved_contexts:
            ctx_words = set(re.findall(r'\w+', ctx.lower()))
            if q_keywords.intersection(ctx_words):
                relevant_contexts += 1
        precision = relevant_contexts / len(retrieved_contexts) if retrieved_contexts else 0.0

        # 2. Faithfulness: How many claims in answer overlap with retrieved context?
        ans_words = set(re.findall(r'\w+', generated_answer.lower()))
        all_ctx_words = set(re.findall(r'\w+', " ".join(retrieved_contexts).lower()))
        supported_words = ans_words.intersection(all_ctx_words)
        faithfulness = len(supported_words) / len(ans_words) if ans_words else 0.0

        # 3. Context Recall: How many ground truth keywords exist in context?
        gt_words = set(re.findall(r'\w+', ground_truth.lower()))
        recalled_gt = gt_words.intersection(all_ctx_words)
        recall = len(recalled_gt) / len(gt_words) if gt_words else 0.0

        return {
            "context_precision": round(precision, 4),
            "faithfulness": round(faithfulness, 4),
            "context_recall": round(recall, 4)
        }

eval_query = "What is RAG?"
retrieved_ctx = [
    "RAG retrieves relevant context from external documents for LLM prompting."
]
gen_ans = "RAG retrieves relevant context from external documents for model prompting."
ground_truth_ans = "RAG retrieves relevant context documents to answer queries."

metrics = RAGEvaluator.evaluate(eval_query, retrieved_ctx, gen_ans, ground_truth_ans)
print("RAG Evaluation Metrics Report:")
for k, v in metrics.items():
    print(f" - {k}: {v * 100:.1f}%")